### Query Translation - Decomposition
Query translation sits at the first stage of an advanced RAG pipeline. The goal of query translation is to take the input user question and to translate it in some
way as to improve retrival.

### What is Query Decomposition?
**Query decomposition** is a technique of breaking one complex user query into **several** _simpler_, _focused_ sub-queries; retrieving for each, and then combining the results. It’s usually done with an LLM:

1. **Input:** a long or multi-part user question.
2. **Decompose:** use an LLM prompt such as
    “Decompose this question into a list of simpler search queries.”
3. **Retrieve:** run each sub-query against your retriever/vector DB.
4. **Synthesize:** feed the retrieved chunks back into the LLM to build the final answer.

#### Example

**User asks:**

    “Summarize Lilian Weng’s post on LLM agents. Focus on task decomposition methods and explain how Tree of Thoughts extends Chain of Thought.”

A **single vector search** might fail because:
* The query contains **multiple sub-topics** (“task decomposition methods” + “Tree of Thoughts vs CoT”).
* The embedding may dilute meaning across the whole sentence.
**Decomposition:**
* “What are task decomposition methods in Lilian Weng’s LLM agents post?”
* “How does Tree of Thoughts extend Chain of Thought reasoning?”

Retrieve separately, then combine into a coherent answer.

### Why / When to Use Query Decomposition

✅ Use it when:
* **Complex / multi-aspect questions:** 
    e.g., “Compare AutoGPT and BabyAGI, and explain how planning differs from memory.”

* **Broad tasks spanning sub-topics:**
    e.g., “Give me the pros/cons of hybrid search and explain when to use reciprocal rank fusion.”

* **Long, natural language queries:** with multiple clauses joined by “and”, “or”, “how … and also …”.

* **Poor retrieval recall:** when a single embedding search often misses pieces of the question.

🚫 Less helpful when:
* The query is **short and atomic** (e.g., “What is RAG Fusion?”).
* The corpus is tiny or each document already covers the entire topic.

In [1]:
import bs4, os
import pathlib
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown

from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# since we are using Gemini, we'll use Google embeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
# load API keys from .env files
load_dotenv(override=True)
# for colorful text output
console = Console()

In [3]:
# create our LLM - we'll be using Gemini-2.5-flash
llm = init_chat_model("google_genai:gemini-2.5-flash", temperature=0.0)
faiss_store = pathlib.Path(os.getcwd()) / "faiss_index_rag_qd"

In [4]:
def create_or_load_embeddings():
    """creates if not available or loads from disk a FAISS embedding"""
    if not faiss_store.exists():
        # in this example we'll load document from a URL
        web_url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
        console.print(
            f"[yellow]Loading document from URL {web_url}. Please wait...[/yellow]"
        )
        loader = WebBaseLoader(
            web_paths=(web_url,),
            bs_kwargs=dict(
                parse_only=bs4.SoupStrainer(
                    class_=("post-content", "post-title", "post-header")
                )
            ),
        )
        blog_docs = loader.load()

        console.print(f"[blue]Loaded {len(blog_docs)} documents from URL[/blue]")
        console.print(
            f"[blue]Metadata of first document: {blog_docs[0].metadata}[/blue]"
        )
        console.print(
            f"[blue]First 200 chars of first document: {blog_docs[0].page_content[:200]}[/blue]"
        )

        # split document into chunks of 1000 chars with 200 chars overlap
        console.print(f"[yellow]Chunking the PDF. Please wait...[/yellow]")

        text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
            chunk_size=300, chunk_overlap=50
        )

        # Make splits
        splits = text_splitter.split_documents(blog_docs)
        console.print(f"[blue]Created {len(splits)} chunks[/blue]")

        # save to embeddings
        console.print("[yellow]Creating embeddings. Please wait...[/yellow]")
        # Use a Gemini embedding model that is suitable for retrieval.
        # It is important to match the model to the task.
        embeddings = GoogleGenerativeAIEmbeddings(
            model="models/text-embedding-004",
            task_type="retrieval_document",
        )
        vector_store = FAISS.from_documents(documents=splits, embedding=embeddings)
        retriever = vector_store.as_retriever()
        vector_store.save_local(str(faiss_store))
        console.print(
            f"[yellow]Local embeddings created at {str(faiss_store)}[/yellow]"
        )
    else:
        console.print(
            f"[yellow]Loading existing embeddings from {str(faiss_store)}[/yellow]"
        )
        embeddings = GoogleGenerativeAIEmbeddings(
            model="models/text-embedding-004",
            task_type="retrieval_document",
        )
        vector_store = FAISS.load_local(
            str(faiss_store), embeddings, allow_dangerous_deserialization=True
        )
        retriever = vector_store.as_retriever()

    return retriever

In [5]:
retriever = create_or_load_embeddings()

Loading existing embeddings from 
c:\Users\BHOBEMRMANISHJAGDISH\Dev\code\git_projects\learning_langchain\src\langchain_tutorial\faiss_index_rag_qd

In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Decomposition
template = """You are a helpful assistant that generates multiple sub-questions related 
to an input question. \n
The goal is to break down the input into a set of sub-problems / sub-questions that 
can be answers in isolation. \n
Generate multiple search queries related to: {question} \n
Generate just the list of queries. Don't generate any other text, such as numbering or additional quotes around the queries or markdown text \n
Output ({num_queries} queries):"""

prompt_decomposition = ChatPromptTemplate.from_template(template)

In [12]:
generate_queries = (
    prompt_decomposition | llm | StrOutputParser() | (lambda x: x.split("\n"))
)

# let's try invoking the chain
questions = generate_queries.invoke(
    {
        "num_queries": 5,
        "question": "What is task decomposition for LLM agents?",
    }
)
console.print(questions)

[
    'definition of task decomposition for LLM agents',
    'why is task decomposition important for LLM agents',
    'methods for task decomposition in LLM agent frameworks',
    'how LLM agents perform task decomposition',
    'examples of task decomposition in autonomous LLM agents'
]

In [13]:
for q in questions:
    print(q)

definition of task decomposition for LLM agents
why is task decomposition important for LLM agents
methods for task decomposition in LLM agent frameworks
how LLM agents perform task decomposition
examples of task decomposition in autonomous LLM agents


So you notice that we generated 5 different versions of the same query to improve our matches against the vector database.

Once we have the decomposed questions, we'll tweak the way the LLM responds to these questions.

#### Answering Recursively
![Multi Query](images/ans_recursively.png)

In [14]:
# Prompt
template = """Here is the question you need to answer:
\n --- \n {question} \n --- \n
Here is any available background question + answer pairs:
\n --- \n {q_a_pairs} \n --- \n
Here is additional context relevant to the question: 
\n --- \n {context} \n --- \n
Use the above context and any background question + answer pairs to answer the question: \n {question}
"""

decomposition_prompt = ChatPromptTemplate.from_template(template)

In [15]:
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser


def format_qa_pair(question, answer):
    """Format Q and A pair"""
    formatted_string = ""
    formatted_string += f"Question: {question}\nAnswer: {answer}\n\n"
    return formatted_string.strip()


q_a_pairs = ""
for q in questions:
    rag_chain = (
        {
            # get the context by asking the retriver to retrieve it 
            # based on the question
            "context": itemgetter("question") | retriever,
            "question": itemgetter("question"),
            # for the first question, q_a_pairs will be ""
            "q_a_pairs": itemgetter("q_a_pairs"),
        }
        # format my prompt with above parameters
        | decomposition_prompt
        # ask LLM for response to formatted decomposition prompt
        | llm
        # parse out text as answer
        | StrOutputParser()
    )
    answer = rag_chain.invoke({"question": q, "q_a_pairs": q_a_pairs})
    q_a_pair = format_qa_pair(q, answer)
    q_a_pairs = q_a_pairs + "\n---\n" + q_a_pair
    console.print(f"[yellow]Intermediate QA-Pair -> [/yellow]")
    console.print(Markdown(q_a_pairs))

Intermediate QA-Pair -> 

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Question: definition of task decomposition for LLM agents Answer: Task decomposition for LLM agents is the process 
of breaking down a large, complicated task into smaller, simpler, and more manageable steps or subgoals. This      
technique is crucial for agents to plan ahead and efficiently handle complex tasks.                                

It is often achieved by:                                                                                           

 1 LLM with simple prompting: Instructing the model to "think step by step" or asking for "subgoals" (e.g., "Steps 
   for XYZ.\n1.", "What are the subgoals for achieving XYZ?"). This is exemplified by Chain of Thought (CoT)       
   prompting, which transforms big tasks into multiple manageable tasks.                                           
 2 Using task-specific instructions: Providing specific guidance for a particular type of task (e.g., "Write a     
   story outline." for writing a novel).                                                                           
 3 Human inputs: Direct human intervention to define the subtasks.                                                 

This process allows the agent to transform complex problems into a series of simpler, executable steps, enabling   
more effective planning and execution.

Intermediate QA-Pair -> 

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Question: definition of task decomposition for LLM agents Answer: Task decomposition for LLM agents is the process 
of breaking down a large, complicated task into smaller, simpler, and more manageable steps or subgoals. This      
technique is crucial for agents to plan ahead and efficiently handle complex tasks.                                

It is often achieved by:                                                                                           

 1 LLM with simple prompting: Instructing the model to "think step by step" or asking for "subgoals" (e.g., "Steps 
   for XYZ.\n1.", "What are the subgoals for achieving XYZ?"). This is exemplified by Chain of Thought (CoT)       
   prompting, which transforms big tasks into multiple manageable tasks.                                           
 2 Using task-specific instructions: Providing specific guidance for a particular type of task (e.g., "Write a     
   story outline." for writing a novel).                                                                           
 3 Human inputs: Direct human intervention to define the subtasks.                                                 


 This process allows the agent to transform complex problems into a series of simpler, executable steps, enabling  
                                      more effective planning and execution.                                       

Question: why is task decomposition important for LLM agents Answer: Task decomposition is crucial for LLM agents  
because it enables them to effectively handle complex tasks by breaking them down into more manageable parts.      

Here's why it's important:                                                                                         

 1 Enables Planning Ahead: A complicated task usually involves many steps. Task decomposition allows the agent to  
   identify these steps and plan ahead, understanding the sequence and requirements for each part.                 
 2 Improves Performance on Complex Tasks: Techniques like Chain of Thought (CoT) demonstrate that instructing the  
   model to "think step by step" by decomposing hard tasks into smaller, simpler steps significantly enhances model
   performance.                                                                                                    
 3 Increases Manageability: It transforms large, daunting tasks into multiple manageable subtasks, making the      
   overall problem more tractable for the LLM agent.                                                               
 4 Facilitates Efficient Handling: By simplifying the problem into smaller subgoals, the agent can process and     
   execute each step more efficiently, leading to more effective task completion.                                  
 5 Supports Advanced Reasoning: It forms the foundation for more sophisticated reasoning strategies, such as Tree  
   of Thoughts, which further explore multiple reasoning possibilities at each decomposed step.                    
 6 Provides Interpretability: Decomposing tasks into steps can shed light on the model's thinking process, making  
   its reasoning more transparent.

Intermediate QA-Pair -> 

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Question: definition of task decomposition for LLM agents Answer: Task decomposition for LLM agents is the process 
of breaking down a large, complicated task into smaller, simpler, and more manageable steps or subgoals. This      
technique is crucial for agents to plan ahead and efficiently handle complex tasks.                                

It is often achieved by:                                                                                           

 1 LLM with simple prompting: Instructing the model to "think step by step" or asking for "subgoals" (e.g., "Steps 
   for XYZ.\n1.", "What are the subgoals for achieving XYZ?"). This is exemplified by Chain of Thought (CoT)       
   prompting, which transforms big tasks into multiple manageable tasks.                                           
 2 Using task-specific instructions: Providing specific guidance for a particular type of task (e.g., "Write a     
   story outline." for writing a novel).                                                                           
 3 Human inputs: Direct human intervention to define the subtasks.                                                 


 This process allows the agent to transform complex problems into a series of simpler, executable steps, enabling  
                                      more effective planning and execution.                                       

Question: why is task decomposition important for LLM agents Answer: Task decomposition is crucial for LLM agents  
because it enables them to effectively handle complex tasks by breaking them down into more manageable parts.      

Here's why it's important:                                                                                         

 1 Enables Planning Ahead: A complicated task usually involves many steps. Task decomposition allows the agent to  
   identify these steps and plan ahead, understanding the sequence and requirements for each part.                 
 2 Improves Performance on Complex Tasks: Techniques like Chain of Thought (CoT) demonstrate that instructing the  
   model to "think step by step" by decomposing hard tasks into smaller, simpler steps significantly enhances model
   performance.                                                                                                    
 3 Increases Manageability: It transforms large, daunting tasks into multiple manageable subtasks, making the      
   overall problem more tractable for the LLM agent.                                                               
 4 Facilitates Efficient Handling: By simplifying the problem into smaller subgoals, the agent can process and     
   execute each step more efficiently, leading to more effective task completion.                                  
 5 Supports Advanced Reasoning: It forms the foundation for more sophisticated reasoning strategies, such as Tree  
   of Thoughts, which further explore multiple reasoning possibilities at each decomposed step.                    
 6 Provides Interpretability: Decomposing tasks into steps can shed light on the model's thinking process, making  
   its reasoning more transparent.                                                                                 

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Question: methods for task decomposition in LLM agent frameworks Answer: Methods for task decomposition in LLM     
agent frameworks include:                                                                                          

 1 LLM with Simple Prompting: This is a common technique where the Large Language Model itself is instructed to    
   break down the task.                                                                                            
    • Chain of Thought (CoT): By prompting the 

Intermediate QA-Pair -> 

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Question: definition of task decomposition for LLM agents Answer: Task decomposition for LLM agents is the process 
of breaking down a large, complicated task into smaller, simpler, and more manageable steps or subgoals. This      
technique is crucial for agents to plan ahead and efficiently handle complex tasks.                                

It is often achieved by:                                                                                           

 1 LLM with simple prompting: Instructing the model to "think step by step" or asking for "subgoals" (e.g., "Steps 
   for XYZ.\n1.", "What are the subgoals for achieving XYZ?"). This is exemplified by Chain of Thought (CoT)       
   prompting, which transforms big tasks into multiple manageable tasks.                                           
 2 Using task-specific instructions: Providing specific guidance for a particular type of task (e.g., "Write a     
   story outline." for writing a novel).                                                                           
 3 Human inputs: Direct human intervention to define the subtasks.                                                 


 This process allows the agent to transform complex problems into a series of simpler, executable steps, enabling  
                                      more effective planning and execution.                                       

Question: why is task decomposition important for LLM agents Answer: Task decomposition is crucial for LLM agents  
because it enables them to effectively handle complex tasks by breaking them down into more manageable parts.      

Here's why it's important:                                                                                         

 1 Enables Planning Ahead: A complicated task usually involves many steps. Task decomposition allows the agent to  
   identify these steps and plan ahead, understanding the sequence and requirements for each part.                 
 2 Improves Performance on Complex Tasks: Techniques like Chain of Thought (CoT) demonstrate that instructing the  
   model to "think step by step" by decomposing hard tasks into smaller, simpler steps significantly enhances model
   performance.                                                                                                    
 3 Increases Manageability: It transforms large, daunting tasks into multiple manageable subtasks, making the      
   overall problem more tractable for the LLM agent.                                                               
 4 Facilitates Efficient Handling: By simplifying the problem into smaller subgoals, the agent can process and     
   execute each step more efficiently, leading to more effective task completion.                                  
 5 Supports Advanced Reasoning: It forms the foundation for more sophisticated reasoning strategies, such as Tree  
   of Thoughts, which further explore multiple reasoning possibilities at each decomposed step.                    
 6 Provides Interpretability: Decomposing tasks into steps can shed light on the model's thinking process, making  
   its reasoning more transparent.                                                                                 

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Question: methods for task decomposition in LLM agent frameworks Answer: Methods for task decomposition in LLM     
agent frameworks include:                                                                                          

 1 LLM with Simple Prompting: This is a common technique where the Large Language Model itself is instructed to    
   break down the task.                                                                                            
    • Chain of Thought (CoT): By prompting the 

Intermediate QA-Pair -> 

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Question: definition of task decomposition for LLM agents Answer: Task decomposition for LLM agents is the process 
of breaking down a large, complicated task into smaller, simpler, and more manageable steps or subgoals. This      
technique is crucial for agents to plan ahead and efficiently handle complex tasks.                                

It is often achieved by:                                                                                           

 1 LLM with simple prompting: Instructing the model to "think step by step" or asking for "subgoals" (e.g., "Steps 
   for XYZ.\n1.", "What are the subgoals for achieving XYZ?"). This is exemplified by Chain of Thought (CoT)       
   prompting, which transforms big tasks into multiple manageable tasks.                                           
 2 Using task-specific instructions: Providing specific guidance for a particular type of task (e.g., "Write a     
   story outline." for writing a novel).                                                                           
 3 Human inputs: Direct human intervention to define the subtasks.                                                 


 This process allows the agent to transform complex problems into a series of simpler, executable steps, enabling  
                                      more effective planning and execution.                                       

Question: why is task decomposition important for LLM agents Answer: Task decomposition is crucial for LLM agents  
because it enables them to effectively handle complex tasks by breaking them down into more manageable parts.      

Here's why it's important:                                                                                         

 1 Enables Planning Ahead: A complicated task usually involves many steps. Task decomposition allows the agent to  
   identify these steps and plan ahead, understanding the sequence and requirements for each part.                 
 2 Improves Performance on Complex Tasks: Techniques like Chain of Thought (CoT) demonstrate that instructing the  
   model to "think step by step" by decomposing hard tasks into smaller, simpler steps significantly enhances model
   performance.                                                                                                    
 3 Increases Manageability: It transforms large, daunting tasks into multiple manageable subtasks, making the      
   overall problem more tractable for the LLM agent.                                                               
 4 Facilitates Efficient Handling: By simplifying the problem into smaller subgoals, the agent can process and     
   execute each step more efficiently, leading to more effective task completion.                                  
 5 Supports Advanced Reasoning: It forms the foundation for more sophisticated reasoning strategies, such as Tree  
   of Thoughts, which further explore multiple reasoning possibilities at each decomposed step.                    
 6 Provides Interpretability: Decomposing tasks into steps can shed light on the model's thinking process, making  
   its reasoning more transparent.                                                                                 

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Question: methods for task decomposition in LLM agent frameworks Answer: Methods for task decomposition in LLM     
agent frameworks include:                                                                                          

 1 LLM with Simple Prompting: This is a common technique where the Large Language Model itself is instructed to    
   break down the task.                                                                                            
    • Chain of Thought (CoT): By prompting the 

In [16]:
console.print(f"Final answer:")
console.print(Markdown(answer))

Final answer:

Autonomous LLM agents like AutoGPT, GPT-Engineer, and BabyAGI are prominent examples that utilize task             
decomposition to handle complex problems.                                                                          

These agents perform task decomposition through various methods:                                                   

 1 Chain of Thought (CoT): The LLM is instructed to "think step by step," which allows it to break down complex    
   tasks into smaller, simpler, and more manageable steps. This is a standard prompting technique to enhance       
   performance.                                                                                                    
 2 Tree of Thoughts (ToT): This method extends CoT by decomposing the problem into multiple thought steps and      
   generating multiple reasoning possibilities at each step, forming a tree structure. A search process (like BFS  
   or DFS) can then explore these paths.                                                                           
 3 LLM with Simple Prompting: The LLM itself can be prompted to generate subtasks using direct questions like      
   "Steps for XYZ.\n1." or "What are the subgoals for achieving XYZ?".                                             
 4 Using Task-Specific Instructions: The agent can be given specific guidance tailored to the task. For instance,  
   for a novel-writing task, the instruction "Write a story outline" guides the decomposition.                     
 5 Human Inputs: Direct human intervention can also define the subtasks or steps for the agent.                    

For example, AutoGPT demonstrates task decomposition by accepting multiple user-provided goals (e.g., "GOALS: 1.   
{{user-provided goal 1}} 2. {{user-provided goal 2}}..."), which serve as initial subtasks for the agent to pursue.